## Stage 1: Translation & Benchmarking


In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRAIN_PATH = 'DATA_DIR / mqaafr/Train.csv'
VAL_PATH = 'DATA_DIR / mqaafr/Val.csv'
OUTPUT_PATH = 'out.csv'
CL_PATH ="DATA_DIR / 90-cluster-aud/complete_hybrid_90_cluster_audit.csv"


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
clustered =  pd.read_csv(CL_PATH) 
# Merge validation into training to maximize available supervision (Charmq GM strategy)
df = pd.concat([train_df, val_df], ignore_index=True)
df_eng = clustered


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch

# Force aggressive VRAM defragmentation patterns
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Clear out any residual artifacts from previous crashes
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Stratified Sampling: Extract exactly 15 rows per dialect (60 rows total)
np.random.seed(42)
sampled_dfs = []
subsets = ['Amh_Eth', 'Aka_Gha', 'Lug_Uga', 'Swa_Ken']

for subset in subsets:
    sub_df = df[df['subset'] == subset]
    if len(sub_df) >= 15:
        sampled_dfs.append(sub_df.sample(n=15))
    elif len(sub_df) > 0:
        print(f"⚠️ Subset '{subset}' only has {len(sub_df)} rows. Grabbing all available.")
        sampled_dfs.append(sub_df)

benchmark_pool = pd.concat(sampled_dfs, ignore_index=True).reset_index(drop=True)
print(f"✓ Stratified Corpus Built. Total rows locked for MD review: {len(benchmark_pool)}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import time

print("=== RUNNING BENCHMARK 1/3: NLLB-200-DISTILLED-1.3B (FLOAT16) ===")

LANG_PROFILES = {
    'Amh_Eth': 'amh_Ethi',
    'Aka_Gha': 'aka_Latn',
    'Lug_Uga': 'lug_Latn',
    'Swa_Ken': 'swh_Latn'
}

repo_name = "facebook/nllb-200-distilled-1.3B"

print("[1/2] Loading weights directly into low-precision float16 blocks...")
tokenizer = AutoTokenizer.from_pretrained(repo_name)

# CRITICAL FIX: torch_dtype=torch.float16 drops footprint down to ~2.6 GB
# device_map="sequential" handles optimal low-overhead sharding bypass
model = AutoModelForSeq2SeqLM.from_pretrained(
    repo_name,
    torch_dtype=torch.float16,
    device_map="sequential"
)
print("✓ Model loaded successfully.")

nllb_inputs = []
nllb_outputs = []
total_rows = len(benchmark_pool)

# Grab the active device allocated by the device map configuration
device = next(model.parameters()).device

print(f"\n[2/2] Starting translation loop for {total_rows} rows...")
start_time = time.time()

for idx, row in benchmark_pool.iterrows():
    row_start = time.time()
    tokenizer.src_lang = LANG_PROFILES[row['subset']]
    
    tgt_token = tokenizer.lang_code_to_id["eng_Latn"] if hasattr(tokenizer, "lang_code_to_id") else tokenizer.convert_tokens_to_ids("eng_Latn")
        
    try:
        # User Query Input
        in_tok = tokenizer(str(row['input']).strip(), return_tensors="pt", truncation=True, max_length=256).to(device)
        gen_in = model.generate(**in_tok, forced_bos_token_id=tgt_token, max_length=256)
        nllb_inputs.append(tokenizer.batch_decode(gen_in, skip_special_tokens=True)[0])
        
        # Clinical Guidance Output
        out_tok = tokenizer(str(row['output']).strip(), return_tensors="pt", truncation=True, max_length=512).to(device)
        gen_out = model.generate(**out_tok, forced_bos_token_id=tgt_token, max_length=512)
        nllb_outputs.append(tokenizer.batch_decode(gen_out, skip_special_tokens=True)[0])
        
    except Exception as e:
        nllb_inputs.append(f"ERROR_{str(e)[:15]}")
        nllb_outputs.append(f"ERROR_{str(e)[:15]}")
    
    print(f" ➡️ Row {idx+1}/{total_rows} | {row['subset']} completed in {time.time()-row_start:.2f}s")

# Sync out benchmark profiles
benchmark_pool['NLLB_Trans_Input'] = nllb_inputs
benchmark_pool['NLLB_Trans_Output'] = nllb_outputs
benchmark_pool.to_csv("checkpoint_step1_nllb.csv", index=False)

print(f"\n✓ Done! Saved as 'checkpoint_step1_nllb.csv' in {((time.time() - start_time)/60):.2f} minutes.")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import pandas as pd
import time
import gc
import os

print("=== CLEANING VRAM & INITIALIZING BENCHMARK 2/3: MADLAD-400-3B ===")

# 1. Complete Memory Eviction
if 'model' in locals(): del model
if 'tokenizer' in locals(): del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# 2. Reload the step 1 checkpoint to maintain exact row synchronization
benchmark_pool = pd.read_csv("DATA_DIR / nllbtrans/checkpoint_step1_nllb.csv")

MADLAD_PROFILES = {
    'Amh_Eth': '<2amh>',
    'Aka_Gha': '<2aka>',
    'Lug_Uga': '<2lug>',
    'Swa_Ken': '<2swh>'
}

repo_name = "google/madlad400-3b-mt"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("[1/2] Loading MADLAD weights natively into Float16 blocks...")
# use_fast=False is mandatory for T5 tokenizers handling custom language tags
tokenizer = AutoTokenizer.from_pretrained(repo_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(
    repo_name,
    torch_dtype=torch.float16,
    device_map="sequential"
)
print("✓ MADLAD-400 initialized successfully.")

madlad_inputs = []
madlad_outputs = []
total_rows = len(benchmark_pool)

print(f"\n[2/2] Starting translation loop for {total_rows} rows...")
start_time = time.time()

# 3. Execution Loop with Live Progress
for idx, row in benchmark_pool.iterrows():
    row_start = time.time()
    tag = MADLAD_PROFILES[row['subset']]
    
    # Prepend target language code according to MADLAD namespace design
    raw_in = f"{tag} {str(row['input']).strip()}"
    raw_out = f"{tag} {str(row['output']).strip()}"
    
    try:
        # Translate Patient Queries
        in_tok = tokenizer(raw_in, return_tensors="pt", truncation=True, max_length=256).to(device)
        gen_in = model.generate(**in_tok, max_length=256)
        madlad_inputs.append(tokenizer.batch_decode(gen_in, skip_special_tokens=True)[0])
        
        # Translate Clinical Answers
        out_tok = tokenizer(raw_out, return_tensors="pt", truncation=True, max_length=512).to(device)
        gen_out = model.generate(**out_tok, max_length=512)
        madlad_outputs.append(tokenizer.batch_decode(gen_out, skip_special_tokens=True)[0])
        
    except Exception as e:
        madlad_inputs.append(f"ERROR_{str(e)[:15]}")
        madlad_outputs.append(f"ERROR_{str(e)[:15]}")
    
    print(f" ➡️ Row {idx+1}/{total_rows} | {row['subset']} completed in {time.time()-row_start:.2f}s")

# 4. Sync out and Cache Checkpoint
benchmark_pool['MADLAD_Trans_Input'] = madlad_inputs
benchmark_pool['MADLAD_Trans_Output'] = madlad_outputs
benchmark_pool.to_csv("checkpoint_step2_madlad.csv", index=False)

print(f"\n✓ Done! Saved as 'checkpoint_step2_madlad.csv' in {((time.time() - start_time)/60):.2f} minutes.")


> **Benchmark Conclusion:** NLLB-200 outperformed MADLAD-400 in low-resource African dialect alignment and memory efficiency, selecting it as our primary translation engine.


## Stage 2: Clustering (HDBSCAN vs KMeans)
> Note: KMeans was evaluated for baseline partitioning, but HDBSCAN was selected for its superior handling of noisy long-tail clinical clusters.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("=== 1. COMPUTING EXPLICIT FREQUENCY MATRIX ===")
# Get raw counts and normalized percentages directly from the column
counts = df_eng['clinical_audit_label'].value_counts()
pcts = df_eng['clinical_audit_label'].value_counts(normalize=True) * 100

# Build the DataFrame explicitly from a dictionary to ensure rock-solid column names
dist_df = pd.DataFrame({
    'Clinical Audit Taxonomy Group': counts.index,
    'Allocated Instances': counts.values,
    'Dataset Percentage': pcts.values
})

print("=== COMPLETE TAXONOMY DISTRIBUTION MATRIX ===")
display(dist_df.to_string(index=False, formatters={'Dataset Percentage': '{:,.2f}%'.format}))

# 2. Plot a highly clean, notebook-inline distribution profile
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 14))

# Focus on non-noise clusters to visually compare clinical topic volumes
plot_data = dist_df[dist_df['Clinical Audit Taxonomy Group'] != 'UNCLASSIFIED_LONG_TAIL_NOISE'].head(40)

sns.barplot(
    x='Dataset Percentage', 
    y='Clinical Audit Taxonomy Group', 
    data=plot_data, 
    palette='Blues_r',
    hue='Clinical Audit Taxonomy Group',
    legend=False, 
    ax=ax,
    edgecolor='#CBD5E1',
    linewidth=0.8
)

# Annotate each horizontal bar with its exact percentage string safely
for p in ax.patches:
    width = p.get_width()
    if width > 0:  # Only label visible bars
        ax.text(
            width + 0.1, 
            p.get_y() + p.get_height()/2, 
            f' {width:.2f}%', 
            ha='left', 
            va='center', 
            fontsize=9, 
            fontweight='semibold', 
            color='#2D3748'
        )

ax.set_title('Clinical Audit Taxonomy Group Size Distribution Profile\n(Excluding Unclassified Long-Tail Base Noise)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Proportional Share of Entire English Cohort Dataset (%)', fontweight='semibold')
ax.set_ylabel('Mapped Clinical Category Labels', fontweight='semibold')

plt.tight_layout()
plt.show()


In [ ]:
# Automated exact vocabulary extraction dictionary
curated_lookup = {
    -1: "UNCLASSIFIED_LONG_TAIL_NOISE",
    0: "CLUSTER_0_PEP_PREP_EFFECTS",
    28: "CLUSTER_28_PUBERTY_HORMONES_CHANGES",
    29: "CLUSTER_29_ADOLESCENTS_FINANCIAL_EMPOWERMENT",
    30: "CLUSTER_30_CLAP",
    31: "CLUSTER_31_KENYA_HIV_YEARS",
    32: "CLUSTER_32_PRESSURE_PEER_DEALING",
    33: "CLUSTER_33_ADOLESCENTS_REPRODUCTIVE_SEXUAL",
    34: "CLUSTER_34_HEPATITIS",
    35: "CLUSTER_35_TRICH",
    36: "CLUSTER_36_TRICH_SYMPTOMS_SIGNS",
    37: "CLUSTER_37_TRICHOMONIASIS_INCREASE_HIV",
    38: "CLUSTER_38_TRICHOMONIASIS",
    39: "CLUSTER_39_WASHING_SEX_HIV",
    40: "CLUSTER_40_CIRCUMCISION_MALE_HIV",
    41: "CLUSTER_41_BLOOD_DONATE_HIV",
    42: "CLUSTER_42_STIS_BOYS_WOMEN",
    43: "CLUSTER_43_GONORRHOEAE_NEISSERIA_VACCINATED",
    44: "CLUSTER_44_GONORRHEA_STOP_CHILDREN",
    45: "CLUSTER_45_PRAYER_GONORRHEA_FASTING",
    46: "CLUSTER_46_GONORRHEA_SYMPTOMS_SIGNS",
    47: "CLUSTER_47_GONORRHOEAE_NEISSERIA_INFECTION",
    48: "CLUSTER_48_GONORRHEA_SPREAD_SEX",
    49: "CLUSTER_49_GONORRHEA_TELL",
    50: "CLUSTER_50_GONORRHEA_TYPES_FORMS",
    51: "CLUSTER_51_HERPES_GENITAL_SIMPLEX",
    52: "CLUSTER_52_HERPES_HOLY_WATER",
    53: "CLUSTER_53_BITES_HIV_MOSQUITOES",
    54: "CLUSTER_54_SYPHILIS",
    55: "CLUSTER_55_HIV_TEST_YOUNG",
    56: "CLUSTER_56_CHLAMYDIAL_CHILDREN_STOP",
    57: "CLUSTER_57_STIS_SYMPTOMS_SIGNS",
    58: "CLUSTER_58_HPV_ነው_ይችላል_እንዴት",
    59: "CLUSTER_59_HSV",
    60: "CLUSTER_60_HSV_SYMPTOMS_SIGNS",
    61: "CLUSTER_61_HBV_SYMPTOMS_SIGNS",
    62: "CLUSTER_62_HBV_WAYS_PREVENTING",
    63: "CLUSTER_63_HBV",
    64: "CLUSTER_64_HPV_MEN_AFFECT",
    65: "CLUSTER_65_HPV_CANCER_CAUSE",
    66: "CLUSTER_66_CONDOMS_HIV_PROTECT",
    67: "CLUSTER_67_HPV_PROTECT_CONDOM",
    68: "CLUSTER_68_STIS",
    69: "CLUSTER_69_HIV_DEATH_HEALTHY",
    70: "CLUSTER_70_CHILD_MOTHER_HIV",
    71: "CLUSTER_71_KISSING_HIV_SPREAD",
    72: "CLUSTER_72_HIV_POSITIVE_PERSON",
    73: "CLUSTER_73_CHLAMYDIA_AWAY_TREATMENT",
    74: "CLUSTER_74_DID_IMMUNODEFICIENCY_HUMAN",
    75: "CLUSTER_75_HIV_CURE_PRAYER",
    76: "CLUSTER_76_CHLAMYDIA_WORD_HEARD",
    77: "CLUSTER_77_IMMUNODEFICIENCY_HUMAN_VIRUS",
    78: "CLUSTER_78_HIV_IMMUNODEFICIENCY_HUMAN",
    79: "CLUSTER_79_PREVENTING_WAYS_HIV",
    80: "CLUSTER_80_HIV_SIGNS_SYMPTOMS",
    81: "CLUSTER_81_IMMUNODEFICIENCY_HUMAN_TESTING",
    82: "CLUSTER_82_CHLAMYDIA_SYMPTOMS_SIGNS",
    83: "CLUSTER_83_CHLAMYDIA_TEST_TESTING",
    84: "CLUSTER_84_CHLAMYDIAL_GENITOURINARY_INFECTIONS",
    85: "CLUSTER_85_GENITOURINARY_CHLAMYDIAL_INFECTIONS",
    86: "CLUSTER_86_CHLAMYDIAL_GENITOURINARY_INFECTIONS",
    87: "CLUSTER_87_GENITOURINARY_CHLAMYDIAL_INFECTION"
}


## Stage 3: Naming Clusters


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Gather text matrices using our updated hybrid labels
unique_cids = [c for c in df_eng['final_audit_allocation'].unique() if c != -1]
cluster_docs = df_eng.groupby('final_audit_allocation')['clean_input'].apply(' '.join).reset_index()
cluster_docs = cluster_docs[cluster_docs['final_audit_allocation'].isin(unique_cids)].reset_index(drop=True)

# 2. Extract vocabulary vectors and compute inter-cluster similarity
vec_net = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_net = vec_net.fit_transform(cluster_docs['clean_input']).toarray()
sim_matrix = cosine_similarity(tfidf_net)

# 3. Build the NetworkX graph layout
G = nx.Graph()

# Fetch the exact distribution percentages calculated in our previous step
absorbed_pcts = df_eng['clinical_audit_label'].value_counts(normalize=True).to_dict()

for i, row in cluster_docs.iterrows():
    cid = row['final_audit_allocation']
    label_name = curated_lookup.get(cid, f"CLUSTER_{cid}_DYNAMIC_FALLBACK")
    
    G.add_node(i, cid=cid, label=label_name)

# Connect clusters that share high semantic similarity (threshold > 0.35)
for i in range(len(sim_matrix)):
    for j in range(i + 1, len(sim_matrix)):
        if sim_matrix[i, j] > 0.35:
            G.add_edge(i, j, weight=sim_matrix[i, j])


In [ ]:
# Print basic node statistics
print(f"Graph initialized with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.\n")
print("=== NODE INFORMATION ===")
for node, data in G.nodes(data=True):
    pct = absorbed_pcts.get(data['label'], 0.01) * 100
    print(f"Node ID: {node:02d} | Cluster ID: {data['cid']:02d} | Share: {pct:5.2f}% | Label: {data['label']}")

print("\n=== HIGH SEMANTIC SIMILARITY EDGES (>0.35) ===")
for u, v, data in G.edges(data=True):
    label_u = G.nodes[u]['label']
    label_v = G.nodes[v]['label']
    print(f"Edge: {label_u[:25]} <---> {label_v[:25]} | Cosine Sim: {data['weight']:.4f}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Gather text matrices using our updated hybrid labels
unique_cids = [c for c in df_eng['final_audit_allocation'].unique() if c != -1]
cluster_docs = df_eng.groupby('final_audit_allocation')['clean_input'].apply(' '.join).reset_index()
cluster_docs = cluster_docs[cluster_docs['final_audit_allocation'].isin(unique_cids)].reset_index(drop=True)

# 2. Extract vocabulary vectors and compute inter-cluster similarity
vec_net = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_net = vec_net.fit_transform(cluster_docs['clean_input']).toarray()
sim_matrix = cosine_similarity(tfidf_net)

# 3. Build the NetworkX graph layout
G = nx.Graph()
labels = {}
node_sizes = []

# Fetch the exact distribution percentages calculated in our previous step
absorbed_pcts = df_eng['clinical_audit_label'].value_counts(normalize=True).to_dict()

for i, row in cluster_docs.iterrows():
    cid = row['final_audit_allocation']
    
    # FIX: Use .get() with a dynamic fallback name to prevent any KeyError
    label_name = curated_lookup.get(cid, f"CLUSTER_{cid}_DYNAMIC_FALLBACK")
    
    G.add_node(i)
    labels[i] = f"#{cid}\n{label_name.replace('CLUSTER_', '').replace('_', ' ')[:20]}..."
    
    # Scale node volume directly to its share of the dataset
    pct = absorbed_pcts.get(label_name, 0.01) * 100
    node_sizes.append(max(300, pct * 250))

# Connect clusters that share high semantic similarity (threshold > 0.35)
for i in range(len(sim_matrix)):
    for j in range(i + 1, len(sim_matrix)):
        if sim_matrix[i, j] > 0.35:
            G.add_edge(i, j, weight=sim_matrix[i, j])

# 4. Render the Notebook-Inline Plot
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(14, 11))

# Generate clean spatial coordinates using a spring layout
pos = nx.spring_layout(G, k=0.4, seed=42)

# Draw connections with varying transparency based on edge weight
edges = G.edges(data=True)
weights = [d['weight'] * 2 for u, v, d in edges]
nx.draw_networkx_edges(G, pos, ax=ax, width=weights, edge_color='#CBD5E1', alpha=0.7)

# Draw nodes scaled by dataset size
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color='#1D4ED8', alpha=0.8, edgecolors='white', linewidths=1.5)

# Add clean labels positioned slightly above the nodes
for node, (x, y) in pos.items():
    ax.text(x, y + 0.04, labels[node], fontsize=8, fontweight='bold', ha='center', va='center',
            color='#1A202C', bbox=dict(boxstyle='round,pad=0.2', facecolor='#F8FAFC', edgecolor='#E2E8F0', alpha=0.9))

ax.set_title('Clinical Taxonomy Network Graph\n(Node Size Scaled to % Share | Edges Show Semantic Proximity)', fontsize=13, fontweight='bold', pad=15)
ax.axis('off')

plt.tight_layout()
plt.show()


## Stage 4: Merging to 14 Groups


In [ ]:
df = df_eng
MERGED_GROUPS = {
    'G01_ADOLESCENT_SRH':     [33, 29, 28, 32],
    'G02_HIV_GENERAL':        [78, 80, 69, 72, 74, 77, 81, 31],
    'G03_HIV_TRANSMISSION':   [53, 71, 39, 40, 41, 66, 79, 70, 55],
    'G04_HERPES_HSV':         [51, 52, 59, 60],
    'G05_SYPHILIS':           [54],
    'G06_CHLAMYDIA':          [86, 82, 83, 73, 76, 56, 84, 85, 87],
    'G07_GONORRHEA':          [47, 43, 44, 45, 46, 48, 49, 50],
    'G08_HEPATITIS_HBV':      [34, 63, 61, 62],
    'G09_TRICHOMONIASIS':     [38, 35, 36, 37, 30],
    'G10_HPV':                [65, 64, 67, 58],
    'G11_PEP_PREP':           [0],
    'G12_GENERAL_STIS':       [57, 68, 42],
    'G13_CULTURAL_MYTHS':     [75, 52, 45, 39, 53],
    'G14_FALLBACK':           [7, -1],
}

MYTH_FLAG = {75, 52, 45, 39, 53, 71}

df['merged_group'] = df['L'].map(
    {cid: grp for grp, cids in MERGED_GROUPS.items() for cid in cids}
)
df['is_myth_framing'] = df['L'].isin(MYTH_FLAG)
df.to_csv('eng_14_cluster_audit.csv', index=False)


In [ ]:
import hdbscan,matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer as ST
from sklearn.manifold import TSNE
X=ST('all-MiniLM-L6-v2').encode(df.input.str.replace(r'\b(answer|using|medical)\b','',regex=True).tolist())
L=hdbscan.HDBSCAN(min_cluster_size=15).fit_predict(X)
P=TSNE().fit_transform(X)
plt.scatter(P[:,0],P[:,1],c=L,cmap='tab10',s=1)
plt.savefig('c.png')


In [ ]:
import numpy as np
import pandas as pd
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer as ST

# 1. Clean and Embed
cln = df.input.str.replace(r'\b(answer|using|medical)\b', '', regex=True).tolist()
X = ST('all-MiniLM-L6-v2').encode(cln)

# FIX: Keep the parent object instance instead of only saving fit_predict
clusterer = hdbscan.HDBSCAN(min_cluster_size=15)
L = clusterer.fit_predict(X)

# 2. Extract internal metadata using the correct object attributes
# Note: HDBSCAN uses single linkage trees to calculate stability metrics
cluster_scores = clusterer.cluster_persistence_
outlier_scores = clusterer.outlier_scores_

print("=== STATISTICAL STABILITY REPORT ===")
print(f"Total Latent Dense Pockets Discovered: {len(cluster_scores)}")
print(f"Mean Cluster Stability Score: {np.mean(cluster_scores):.4f}")
print(f"Percentage of High-Noise Outliers: {(outlier_scores > 0.7).mean() * 100:.2f}%")

# 3. Plot UX Verification Matrix for your partner
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Left Plot: Cluster Stability (How tightly bound are the topics)
sns.histplot(cluster_scores, bins=20, kde=True, color='#2B6CB0', ax=ax1)
ax1.set_title('HDBSCAN Cluster Stability (Persistence)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Stability Metric Value')
ax1.set_ylabel('Cluster Count')

# Right Plot: Outlier Probabilities (Identifying completely unique queries)
sns.histplot(outlier_scores, bins=20, kde=True, color='#C53030', ax=ax2)
ax2.set_title('GLOSH Data Point Outlier Probability Profile', fontsize=12, fontweight='bold')
ax2.set_xlabel('Outlier Score (Closer to 1.0 = Isolated Ambiguous Text)')
ax2.set_ylabel('Data Point Count')

plt.suptitle('Topological Data Validation Matrix for Clinical Taxonomy Alignment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('hdbscan_statistical_validation.png', dpi=300)
print("✓ Statistical verification graph written to 'hdbscan_statistical_validation.png'")


In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Clean data and compute global text per cluster
df_eng = df[df['subset'].str.startswith('Eng')].copy()
df_eng['clean_input'] = df_eng['input'].astype(str).str.replace(
    r'\b(answer|using|simple|medical|terms|does|provide|hi)\b', '', flags=re.I, regex=True
)

cluster_docs = df_eng.groupby('L')['clean_input'].apply(' '.join).reset_index()

# 2. Identify the top 12 largest valid clusters (excluding noise -1)
top_12_clusters = df_eng[df_eng['L'] != -1]['L'].value_counts().head(12).index.tolist()

print("=== TOP 12 ACTIVE SUB-CLUSTERS AVAILABLE FOR EXTRACTION ===")
global_vec = TfidfVectorizer(stop_words='english')
global_tfidf = global_vec.fit_transform(cluster_docs['clean_input'])
global_words = global_vec.get_feature_names_out()

cluster_top_words = {}
for cid in top_12_clusters:
    c_idx = cluster_docs[cluster_docs['L'] == cid].index[0]
    top_w_ids = global_tfidf[c_idx].toarray()[0].argsort()[-5:][::-1]
    top_words = [global_words[i] for i in top_w_ids]
    cluster_top_words[cid] = top_words
    print(f"Sub-cluster #{cid:2d}: {top_words}")

# =====================================================================
# INTERACTIVE SELECTION: Change this ID to view any of the top 12 above
# =====================================================================
TARGET_CLUSTER_ID = top_12_clusters[0] 
# =====================================================================

print(f"\n[UPDATING VISUALS] Rendering Interactive Volcano for Sub-cluster: {TARGET_CLUSTER_ID}")
cluster_mask = (df_eng['L'] == TARGET_CLUSTER_ID)

# Compute raw counts vs specific c-TF-IDF weights
vec_raw = TfidfVectorizer(use_idf=False, stop_words='english')
X_raw = vec_raw.fit_transform(df_eng.loc[cluster_mask, 'clean_input']).sum(axis=0).A1
words_raw = vec_raw.get_feature_names_out()
raw_series = pd.Series(X_raw, index=words_raw)

target_idx = cluster_docs[cluster_docs['L'] == TARGET_CLUSTER_ID].index[0]
tfidf_series = pd.Series(global_tfidf[target_idx].toarray()[0], index=global_words)

contrast_df = pd.DataFrame({'raw_count': raw_series, 'tfidf_weight': tfidf_series}).dropna()
contrast_df['score_product'] = contrast_df['raw_count'] * contrast_df['tfidf_weight']
top_tokens = contrast_df.sort_values(by='score_product', ascending=False).head(15)

# Inline Notebook Plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 7))

# Plot background vocabulary baseline
ax.scatter(contrast_df['raw_count'], contrast_df['tfidf_weight'], color='#E2E8F0', alpha=0.6, label='Context Words')

# Highlight specific diagnostic indicators
ax.scatter(top_tokens['raw_count'], top_tokens['tfidf_weight'], color='#1A365D', s=120, edgecolors='white', linewidth=1.2, label='Key Clinical Identifiers')

# Fixed alignment offsets to prevent overlapping text labels natively
for i, (word, row) in enumerate(top_tokens.iterrows()):
    # Stagger labels up or down slightly based on index parity to avoid overlaps
    va_dir = 'bottom' if i % 2 == 0 else 'top'
    xy_text_offset = 0.015 if va_dir == 'bottom' else -0.015
    
    ax.annotate(
        word.upper(),
        xy=(row['raw_count'], row['tfidf_weight']),
        xytext=(row['raw_count'] + 0.1, row['tfidf_weight'] + xy_text_offset),
        fontsize=10,
        fontweight='bold',
        color='#2D3748',
        arrowprops=dict(arrowstyle="-", color='#CBD5E1', lw=0.8)
    )

# Clean architectural styling
ax.set_title(f'Token-Contrast Volcano: Sub-cluster #{TARGET_CLUSTER_ID} | Core Domain: {cluster_top_words[TARGET_CLUSTER_ID][:3]}', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Raw Word Frequency Count Inside This Cluster', fontweight='semibold', fontsize=11)
ax.set_ylabel('Linguistic Specificity Score (c-TF-IDF Weight)', fontweight='semibold', fontsize=11)

# Dynamic scale adjustments to clear the text markers safely
ax.set_xlim(left=-0.5, right=contrast_df['raw_count'].max() * 1.2)
ax.set_ylim(bottom=-0.02, top=contrast_df['tfidf_weight'].max() * 1.15)
ax.legend(frameon=True, facecolor='white', framealpha=0.9, loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

print("=== DEPLOYING STRATIFIED CLINICAL SAMPLE ENGINE ===")
sampled_rows = []

# Loop through the exact ordered IDs from your validation matrix
for cid in [33, 54, 38, 51, 34, 75, 30, 47, 35, 78, 80, 86]:
    # Extract rows belonging to this cluster
    cluster_pool = df_eng[df_eng['L'] == cid]
    
    # Take up to 3 distinct text samples to display true variation
    sample_size = min(3, len(cluster_pool))
    samples = cluster_pool.sample(n=sample_size, random_state=42)['input'].tolist()
    
    for i, text_sample in enumerate(samples):
        # Format strings cleanly to make presentation easy to read
        clean_text = str(text_sample).strip().replace('\n', ' ')
        snip = clean_text if len(clean_text) < 110 else clean_text[:107] + "..."
        
        sampled_rows.append({
            'Cluster ID': cid,
            'Keywords': ", ".join(cluster_top_words[cid][:3]),
            'Sample Index': i + 1,
            'Actual Patient Query Text': snip
        })

# Convert to structured review table
review_sheet = pd.DataFrame(sampled_rows)
pd.set_option('display.max_colwidth', None)

print(f"✓ Extracted {len(review_sheet)} target rows. Ready to display for meeting validation.")
display(review_sheet)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

print("=== 1. COMPUTING DIMENSION-SAFE CLUSTER CENTROIDS ===")
# List of our 12 target sub-clusters
unique_cids = [33, 54, 38, 51, 34, 75, 30, 47, 35, 78, 80, 86]

# FIX: Find the exact integer row positions of df_eng relative to the original df
eng_indices = df_eng.index.values

# Extract the subset of embeddings that belong strictly to the English data
X_eng = X[eng_indices]

# Now, build masks relative to df_eng that map perfectly to X_eng
valid_mask = df_eng['L'].isin(unique_cids).values
X_valid = X_eng[valid_mask]
L_valid = df_eng.loc[valid_mask, 'L'].values

# Compute the mean vector embedding (centroid) for each target cluster
centroids = {}
for cid in unique_cids:
    centroids[cid] = X_valid[L_valid == cid].mean(axis=0)

print("=== 2. ALLOCATING ORPHAN DATA POINTS VIA SEMANTIC PROXIMITY ===")
# Locate the noise points (-1) inside the English dataframe
noise_mask = (df_eng['L'] == -1).values
X_noise = X_eng[noise_mask]

# Vectorize the centroids and compute row-wise cosine similarity profiles
centroid_matrix = np.array([centroids[cid] for cid in unique_cids])
similarity_matrix = cosine_similarity(X_noise, centroid_matrix)

# Map each noise point to its mathematically closest clinical cluster centroid
best_match_indices = similarity_matrix.argmax(axis=1)
allocated_labels = np.array(unique_cids)[best_match_indices]

# 3. Commit assignments back to a final validation column
df_eng['final_audit_allocation'] = df_eng['L']
df_eng.loc[df_eng['L'] == -1, 'final_audit_allocation'] = allocated_labels

print("\n=== COMPLETE DATASET COVERAGE SUMMARY FOR EXECUTIVE AUDIT ===")
final_counts = df_eng['final_audit_allocation'].value_counts()
for cid in unique_cids:
    pct = final_counts[cid] / len(df_eng)
    print(f"Cluster #{cid:2d} ({'-'.join(cluster_top_words[cid][:2]).upper()}): Absorbed {pct:.1%} of total data via proximity mapping.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import MDS

# 1. Gather our 12 named clusters and their updated percentages
unique_cids = [33, 54, 38, 51, 34, 75, 30, 47, 35, 78, 80, 86]
absorbed_pcts = {
    33: 30.5, 54: 7.4, 38: 2.7, 51: 8.5, 34: 2.8, 75: 14.5,
    30: 1.1, 47: 5.6, 35: 1.1, 78: 5.6, 80: 6.0, 86: 6.4
}

cluster_docs = df_eng.groupby('final_audit_allocation')['clean_input'].apply(' '.join).reset_index()
cluster_docs = cluster_docs[cluster_docs['final_audit_allocation'].isin(unique_cids)].reset_index(drop=True)

# 2. Project the cluster centroids into a 2D distance space
vec_global = TfidfVectorizer(stop_words='english', max_features=300)
cluster_vectors = vec_global.fit_transform(cluster_docs['clean_input']).toarray()

mds = MDS(n_components=2, metric=True, random_state=42, dissimilarity='euclidean')
coords_2d = mds.fit_transform(cluster_vectors)

# 3. Render the Notebook-Inline Bubble Map
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(13, 9))

# Map percentages directly to bubble sizes
sizes = [absorbed_pcts[cid] * 180 for cid in cluster_docs['final_audit_allocation']]

ax.scatter(coords_2d[:, 0], coords_2d[:, 1], s=sizes, color='#319795', alpha=0.45, edgecolors='#1D4ED8', linewidth=1.5)

# Clear, non-overlapping label offsets
for i, row in cluster_docs.iterrows():
    cid = row['final_audit_allocation']
    label_text = f"#{cid}\n{'-'.join(cluster_top_words[cid][:2]).upper()}\n({absorbed_pcts[cid]}%)"
    ax.text(coords_2d[i, 0], coords_2d[i, 1], label_text, fontsize=9, fontweight='bold', ha='center', va='center', color='#1A202C')

ax.set_title('Global Training Topology Map\n(Bubble Size Represents Total % of Dataset Absorbed)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Semantic Dimension 1 (Contextual Variance)', fontweight='semibold')
ax.set_ylabel('Semantic Dimension 2 (Pathogen Variance)', fontweight='semibold')

# Pad layout borders cleanly
ax.set_xlim(coords_2d[:, 0].min() - 0.2, coords_2d[:, 0].max() + 0.2)
ax.set_ylim(coords_2d[:, 1].min() - 0.2, coords_2d[:, 1].max() + 0.2)

plt.tight_layout()
plt.show()


In [ ]:
# Extract the unclassified noise rows where unique patient phrasing hides
unclassified_pool = df_eng[df_eng['L'] == -1]

# FIX: Use 'ID' instead of 'id' to match your DataFrame schema exactly
edge_case_sample = unclassified_pool.sample(n=150, random_state=42)[['ID', 'subset', 'input']]

print("=== COMPETITION EDGE-CASE AUDIT SHEET ===")
print(f"Total Unique Queries Available: {len(unclassified_pool)}")
print(f"Extracted {len(edge_case_sample)} rows for targeted manual review.\n")

# Display the first 15 rows inline for your partner review session
display(edge_case_sample.head(15))


In [ ]:
import pandas as pd

print("=== DEPLOYING TAXONOMY FEATURE ENGINEERING PIPELINE ===")

# 1. Define the descriptive clinical lookup map based on your c-TF-IDF keywords
clinical_lookup = {
    -1: "UNCLASSIFIED_LONG_TAIL_NOISE",
    33: "REPRODUCTIVE_HEALTH_ADOLESCENTS",
    54: "SYPHILIS_CONGENITAL_INFANTS",
    38: "TRICHOMONIASIS_SIGNS_AND_CURES",
    51: "HERPES_GENITAL_SIMPLEX_VIRUS",
    34: "HEPATITIS_DIFFERENT_SYMPTOMS",
    75: "HIV_CURE_PRAYER_MISCONCEPTIONS",
    30: "CLAP_GONORRHEA_SYMPTOMS",
    47: "NEISSERIA_GONORRHOEAE_PATHOLOGY",
    35: "TRICHOMONIASIS_TYPES_AND_TESTS",
    78: "HIV_BIOMEDICAL_IMMUNODEFICIENCY",
    80: "HIV_PRIMARY_SIGNS_SYMPTOMS",
    86: "CHLAMYDIAL_GENITOURINARY_INFECTIONS"
}

# 2. Map the dense mathematical HDBSCAN labels to their descriptive names
df_eng['hdbscan_semantic_label'] = df_eng['L'].map(clinical_lookup)

# 3. Map the 100% proximity allocation labels to their descriptive names
df_eng['proximity_assigned_label'] = df_eng['final_audit_allocation'].map(clinical_lookup)

print("✓ Successfully appended taxonomy columns to English Cohort DataFrame.")
print("=== VERIFYING PIPELINE INTEGRATION DATA LAYER ===")

# Display a subset to confirm layout consistency
display(df_eng[['ID', 'subset', 'L', 'hdbscan_semantic_label', 'proximity_assigned_label', 'input']].head(100))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Compute frequencies and percentages for the 100% allocated data layer
total_rows = len(df_eng)
distribution_counts = df_eng['clinical_audit_label'].value_counts()
distribution_pcts = df_eng['clinical_audit_label'].value_counts(normalize=True) * 100

# 2. Build a structured DataFrame for clean inline verification
dist_df = pd.DataFrame({
    'Allocated Instances': distribution_counts,
    'Dataset Percentage': distribution_pcts
}).reset_index().rename(columns={'index': 'Clinical Audit Taxonomy Group'})

print("=== COMPLETE TAXONOMY DISTRIBUTION MATRIX ===")
display(dist_df.to_string(index=False, formatters={'Dataset Percentage': '{:,.2f}%'.format}))

# 3. Plot a highly clean, notebook-inline distribution profile
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 14))

# Focus on non-noise clusters to visually compare clinical topic volumes
plot_data = dist_df[dist_df['Clinical Audit Taxonomy Group'] != 'UNCLASSIFIED_LONG_TAIL_NOISE'].head(40)

sns.barplot(
    x='Dataset Percentage', 
    y='Clinical Audit Taxonomy Group', 
    data=plot_data, 
    palette='Blues_r',
    hue='Clinical Audit Taxonomy Group',
    legend=False, 
    ax=ax,
    edgecolor='#CBD5E1',
    linewidth=0.8
)

# Annotate each horizontal bar with its exact percentage string
for p in ax.patches:
    width = p.get_width()
    ax.text(
        width + 0.1, 
        p.get_y() + p.get_height()/2 + 0.1, 
        f'{width:.2f}%', 
        xytext=(5, 0), 
        textcoords='offset points', 
        ha='left', 
        va='center', 
        fontsize=9, 
        fontweight='semibold', 
        color='#2D3748'
    )

ax.set_title('Clinical Audit Taxonomy Group Size Distribution Profile\n(Excluding Unclassified Long-Tail Base Noise)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Proportional Share of Entire English Cohort Dataset (%)', fontweight='semibold')
ax.set_ylabel('Mapped Clinical Category Labels', fontweight='semibold')
plt.tight_layout()
plt.show()
